# 【实战】使用 GRPO 强化模型的工具调度能力

## 任务背景

在复杂数学计算场景中，模型不仅需要理解算式的语义，更需要高效调度外部工具来辅助求解。然而，经过监督微调（Supervised Fine-Tuning，SFT）的模型虽然能遵循指令并生成格式规范的中间步骤，却往往只会机械地按顺序串行调用工具，难以根据算式的内在结构灵活编排调度策略。面对 `(3 + 5) × (8 - 2) ÷ 4` 这类包含多层括号和混合运算符的表达式，SFT模型通常会从左到右依次执行，完全忽略了不同子表达式之间天然存在的并行执行潜力，导致响应延迟增加，且因中间步骤堆积而更容易出错。

强化学习为训练“智能调度策略”提供了有效路径。传统PPO（Proximal Policy Optimization）算法需要额外维护庞大的Critic网络来评估状态价值，在工具调度这种动作空间庞大且状态转移复杂的场景中，训练开销令人望而却步。

GRPO（Group Relative Policy Optimization）则通过群组采样和组内相对优势估计，摒弃了Critic模型，大幅降低了资源门槛。尤其值得强调的是，工具调度的成败可以直接通过最终计算结果来验证，这让GRPO成为理想之选——奖励函数可以基于计算结果正确性直接定义，无需训练额外的偏好模型。DeepSeek-Math等前沿工作已经证明，GRPO能有效驱动模型从“被动执行”进化为“主动规划”，在复杂运算中实现更优的调度策略。

## 任务目标

本次任务将基于GRPO强化学习算法，训练模型掌握加减乘除及括号的运算优先级规则，核心目标是让模型学会**根据算式算子的优先级动态规划并行调度策略**。具体而言，模型需要做到以下四点：
1. **深度解析算式结构**：对于给定的复杂数学表达式，模型能够准确解析其层级结构，识别出各级运算符（括号、乘除、加减）之间的依赖关系和优先级约束，构建出完整的计算依赖图。
2. **动态规划调度策略**：基于解析出的运算符优先级，模型自主规划并行执行方案——将不存在数据依赖关系的子表达式识别为可并行节点，设计出最优的调度时序图，明确哪些子任务可以同时执行、哪些必须等待前置结果。
3. **灵活并行调度多工具**：根据规划好的调度策略，模型为每个可并行执行的子任务分配最合适的计算工具，并一次性或分批发起多工具并行调用，最大化利用计算资源，显著缩短整体求解时间。
4. **聚合结果并完成求解**：在获取所有并行子任务的返回结果后，模型严格按照原始算式的优先级结构将这些中间结果拼接组合，最终输出正确的完整计算结果。

通过GRPO训练，模型将学会根据算式的复杂度、运算符层级和子表达式的独立性，自主权衡“串行依赖”与“并行执行”的边界，动态生成任务调度图，在保证准确率的前提下追求极致的计算效率。

## 学习收获

完成本次任务，你将能够：
- **理解 GRPO 在动态并行调度中的适用性**：掌握GRPO如何通过组内相对优势比较来引导模型优化调度策略，理解其与PPO在训练开销上的本质差异，并明确为什么工具调度这类“结果可验证”的任务特别适合GRPO——奖励可直接基于最终计算结果正误定义，无需额外训练奖励模型。
- **明确 GRPO 训练的数据与奖励设计需求**：熟悉工具调度任务所需的数据格式，掌握如何设计复合奖励函数——例如，**结果奖励**（最终结果正确得高分）+ **调度效率奖励**（鼓励更短的并行执行步数）+ **格式规范奖励**（确保工具调用序列合法），并理解如何避免模型为追求并行而生成错误的调度顺序。
- **掌握 GRPO 的标准训练流程**：理解三个核心阶段：
	- **Rollout阶段**：对同一算式并行采样多个候选调度方案（包含工具分配序列及预估执行时序）；
	- **奖励计算阶段**：根据每个方案的计算正确性、并行效率给出综合评分，并在组内归一化得到相对优势；
	- **策略优化阶段**：利用组内相对优势更新策略模型，促使模型逐步学会根据优先级动态生成更优的并行调度策略。
- **了解 GRPO 训练中的关键监控指标**：通过追踪**奖励均值**（反映整体调度准确率与效率）、**并行调用比例**（反映模型是否充分利用并行机会）、**平均调度步数**（反映从串行到并行的优化程度）以及**调度方案多样性**（反映模型是否陷入局部最优），判断训练是否朝着动态规划更优并行策略的方向稳定收敛，并适时调整超参数。

## 获取源码

点击下方链接，获取任务完整源码 👇👇👇

```{div} alert alert-light simple-tilt custom-card

<a href="https://note.mowen.cn/intro/9PqPCMHa1UTFRuV5JxsDL" onclick="LA.track('dataset'); return true;">
<div>
<div style="display: flex; align-items: center">
<img src="https://fasterai-picgo.oss-cn-beijing.aliyuncs.com/logo.png" style="width: 128px">
<div style="margin-left: 20px">
<h4 class="card-title mt-3">《动手学大模型：实战进阶》</h4>
<p class="card-text">
通过 30 个动手实战任务，将传统 2-3 年的大模型学习周期压缩至 2-3 个月，助你系统掌握大模型知识，快速拿到理想Offer 🎉
</p>
</div>
</div>
</div>
</a>
```